# BERTopic Exploration - 8 Topics with Offline LLM Labeling

This notebook uses **offline models** for accurate topic labeling with a fixed number of 8 topics.

**Models used:**
- **Primary**: Flan-T5-Large (reliable, ~3GB)
- **Alternative**: Llama-3.2-1B-Instruct or Phi-3-mini

All models run locally on your hardware (GPU/CPU) with **no API costs**.

**Key Features:**
- **Fixed 8 topics**: Clear, manageable number of categories
- **Offline LLM labeling**: High-quality topic names
- **Reproducible**: Deterministic with seeds
- **Optimized for Apple Silicon (MPS) and CUDA**

In [1]:
# Import necessary libraries
import pandas as pd
import numpy as np
from bertopic import BERTopic
import matplotlib.pyplot as plt
import seaborn as sns
import random
import os
import re
from typing import List, Tuple, Dict, Optional
from collections import Counter
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM, pipeline

# Set seeds for reproducibility
SEED = 42
random.seed(SEED)
np.random.seed(SEED)
os.environ['PYTHONHASHSEED'] = str(SEED)

# Set PyTorch seed if available
try:
    import torch
    torch.manual_seed(SEED)
    if torch.cuda.is_available():
        torch.cuda.manual_seed(SEED)
        torch.cuda.manual_seed_all(SEED)
    if torch.backends.mps.is_available():
        torch.mps.manual_seed(SEED)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False
except ImportError:
    pass

# Set display options
pd.set_option('display.max_columns', None)
pd.set_option('display.max_colwidth', None)

print(f"✅ Random seed set to {SEED} for reproducibility")
print("All random number generators have been seeded.")

✅ Random seed set to 42 for reproducibility
All random number generators have been seeded.


## Helper Functions

In [ ]:
def clean_text(text: str) -> str:
    """Clean individual text response."""
    text = str(text).strip()
    text = re.sub(r'\s+', ' ', text)
    return text

def preprocess_responses(responses: List[str],
                        min_length: int = 2,
                        remove_duplicates: bool = False,
                        verbose: bool = True) -> Tuple[List[str], Dict]:
    """Preprocess survey responses with cleaning and filtering."""
    original_count = len(responses)
    stats = {'original_count': original_count}
    
    cleaned = [clean_text(r) for r in responses]
    cleaned = [r for r in cleaned if r and r.lower() not in ['', 'nan', 'none', 'n/a']]
    stats['after_empty_removal'] = len(cleaned)
    stats['empty_removed'] = original_count - len(cleaned)
    
    length_filtered = [r for r in cleaned if len(r.split()) >= min_length]
    stats['after_length_filter'] = len(length_filtered)
    stats['short_removed'] = len(cleaned) - len(length_filtered)
    
    if remove_duplicates:
        before_dedup = len(length_filtered)
        length_filtered = list(dict.fromkeys(length_filtered))
        stats['after_deduplication'] = len(length_filtered)
        stats['duplicates_removed'] = before_dedup - len(length_filtered)
    
    stats['final_count'] = len(length_filtered)
    
    if verbose:
        print("=" * 60)
        print("DATA PREPROCESSING SUMMARY")
        print("=" * 60)
        print(f"Original responses: {stats['original_count']}")
        print(f"Empty/invalid removed: {stats['empty_removed']}")
        print(f"Too short (< {min_length} words) removed: {stats['short_removed']}")
        if remove_duplicates:
            print(f"Duplicates removed: {stats['duplicates_removed']}")
        print(f"\n✅ Final response count: {stats['final_count']}")
        print(f"   Retention rate: {stats['final_count']/stats['original_count']*100:.1f}%")
        print("=" * 60)
    
    return length_filtered, stats

def analyze_response_statistics(responses: List[str]) -> None:
    """Print detailed statistics about the responses."""
    word_counts = [len(r.split()) for r in responses]
    char_counts = [len(r) for r in responses]
    
    print("\n📊 Response Statistics:")
    print(f"   Total responses: {len(responses)}")
    print(f"\n   Word count per response:")
    print(f"      Mean: {np.mean(word_counts):.1f}")
    print(f"      Median: {np.median(word_counts):.0f}")
    print(f"      Min: {np.min(word_counts)}")
    print(f"      Max: {np.max(word_counts)}")
    print(f"\n   Character count per response:")
    print(f"      Mean: {np.mean(char_counts):.1f}")
    print(f"      Median: {np.median(char_counts):.0f}")
    
    duplicates = pd.Series(responses).value_counts()
    if (duplicates > 1).any():
        print(f"\n   ⚠️ Found {(duplicates > 1).sum()} duplicate responses:")
        for resp, count in duplicates[duplicates > 1].head(5).items():
            print(f"      '{resp[:50]}...' appears {count} times")
    else:
        print(f"\n   ✅ No duplicate responses found")

def setup_device() -> Tuple[str, int]:
    """Detect and configure the best available device."""
    try:
        import torch
        if torch.backends.mps.is_available():
            print("✅ Using Apple Silicon GPU (Metal) for acceleration")
            os.environ['PYTORCH_ENABLE_MPS_FALLBACK'] = '1'
            return "mps", 0
        elif torch.cuda.is_available():
            print(f"✅ Using NVIDIA CUDA GPU: {torch.cuda.get_device_name(0)}")
            return "cuda", 0
    except ImportError:
        pass
    
    print("⚠️ Using CPU (no GPU detected - this will be slower)")
    return "cpu", -1

def load_llama_chat_model(model_name: str = "meta-llama/Llama-3.2-1B-Instruct", 
                          device: str = "auto",
                          seed: int = 42):
    """Load Llama model with chat template support for topic labeling.
    
    Args:
        model_name: HuggingFace model identifier
        device: Device to use ("auto", "mps", "cuda", or "cpu")
        seed: Random seed for reproducibility
        
    Returns:
        Tuple of (tokenizer, model, device_str)
    """
    from transformers import AutoTokenizer, AutoModelForCausalLM, set_seed
    import torch
    
    set_seed(seed)
    
    if device == "auto":
        if torch.backends.mps.is_available():
            device = "mps"
            os.environ['PYTORCH_ENABLE_MPS_FALLBACK'] = '1'
        elif torch.cuda.is_available():
            device = "cuda"
        else:
            device = "cpu"
    
    print(f"\n🔄 Loading {model_name}...")
    print(f"   Device: {device}")
    
    tokenizer = AutoTokenizer.from_pretrained(model_name)
    model = AutoModelForCausalLM.from_pretrained(
        model_name,
        torch_dtype=torch.float16 if device in ["cuda", "mps"] else torch.float32,
        device_map=device if device != "mps" else None
    )
    
    if device == "mps":
        model = model.to("mps")
    
    print(f"✅ Successfully loaded {model_name}")
    print(f"   Model stored locally in HuggingFace cache")
    
    return tokenizer, model, device

def generate_topic_label_with_llama(tokenizer, model, keywords: List[str], 
                                    examples: List[str], max_new_tokens: int = 50):
    """Generate a topic label using Llama with chat template.
    
    Args:
        tokenizer: Llama tokenizer
        model: Llama model
        keywords: List of topic keywords
        examples: List of example documents
        max_new_tokens: Maximum tokens to generate
        
    Returns:
        Generated topic label
    """
    import torch
    
    keywords_str = ", ".join(keywords[:15])  # Use all 15 keywords for more context
    examples_str = "\n".join([f"- {ex[:150]}" for ex in examples[:8]])  # More examples, longer snippets
    
    prompt = f"""You are analyzing survey responses about fishing site preferences.

Topic Keywords: {keywords_str}

Example Responses:
{examples_str}

Task: Create a descriptive, specific label (5-15 words) that captures the MAIN theme and context of this topic.

Guidelines:
- Be SPECIFIC and DESCRIPTIVE, not generic (e.g., "Boat Launch Facilities and Access Points" not "Fishing")
- Include relevant context from the keywords and examples
- Focus on what makes this topic UNIQUE from other topics
- Use clear, professional language
- NO quotes, NO explanations, ONLY the label
- The label should be informative enough to understand the topic's main characteristics

Label:"""
    
    messages = [{"role": "user", "content": prompt}]
    
    inputs = tokenizer.apply_chat_template(
        messages,
        add_generation_prompt=True,
        tokenize=True,
        return_dict=True,
        return_tensors="pt",
    ).to(model.device)
    
    with torch.no_grad():
        outputs = model.generate(**inputs, max_new_tokens=max_new_tokens, do_sample=False)
    
    generated_text = tokenizer.decode(
        outputs[0][inputs["input_ids"].shape[-1]:],
        skip_special_tokens=True
    ).strip()
    
    return generated_text

def setup_offline_llm(model_choice: str = "auto", seed: int = 42):
    """Setup the best available offline LLM for topic labeling."""
    from transformers import pipeline, set_seed
    from bertopic.representation import TextGeneration
    
    set_seed(seed)
    device_name, device_id = setup_device()
    
    models = {
        "llama": {
            "name": "meta-llama/Llama-3.2-1B-Instruct",
            "task": "text-generation",
            "max_length": 100,
            "description": "Meta Llama 3.2 1B (best quality, ~5GB)",
            "tokenizer": None
        },
        "phi": {
            "name": "microsoft/Phi-3-mini-4k-instruct",
            "task": "text-generation",
            "max_length": 100,
            "description": "Microsoft Phi-3 Mini (fast, ~7GB)",
            "tokenizer": None
        },
        "flan-t5": {
            "name": "google/flan-t5-large",
            "task": "text2text-generation",
            "max_length": 50,
            "description": "Google Flan-T5 Large (reliable, ~3GB)",
            "tokenizer": None
        },
        "flan-t5-base": {
            "name": "google/flan-t5-base",
            "task": "text2text-generation",
            "max_length": 50,
            "description": "Google Flan-T5 Base (fast, ~1GB)",
            "tokenizer": None
        }
    }
    
    model_order = ["flan-t5", "flan-t5-base", "llama", "phi"] if model_choice == "auto" else [model_choice]
    
    generator = None
    selected_model = None
    
    for model_key in model_order:
        if model_key not in models:
            continue
        
        model_config = models[model_key]
        
        try:
            print(f"\n🔄 Loading {model_config['description']}...")
            
            generator = pipeline(
                model_config['task'],
                model=model_config['name'],
                device=device_id,
                dtype=torch.float16,
                max_length=model_config['max_length'],
                truncation=True
            )
            
            selected_model = model_key
            print(f"✅ Successfully loaded {model_config['name']}")
            break
        
        except Exception as e:
            print(f"⚠️ Failed to load {model_key}: {str(e)}")
            if model_choice != "auto":
                raise
            continue
    
    if generator is None:
        raise RuntimeError("Failed to load any offline LLM model")
    
    selected_config = models[selected_model]
    
    if selected_model in ["llama", "phi"]:
        prompt = """You are analyzing survey responses about fishing site preferences.
Topic Keywords: [KEYWORDS]
Example Responses:
[DOCUMENTS]

Task: Create a descriptive, specific label (5-15 words) that captures the MAIN theme and context of this topic.

Guidelines:
- Be SPECIFIC and DESCRIPTIVE, not generic (e.g., "Boat Launch Facilities and Access Points" not "Fishing")
- Include relevant context from the keywords and examples
- Focus on what makes this topic UNIQUE from other topics
- Use clear, professional language
- NO quotes, NO explanations, ONLY the label
- The label should be informative enough to understand the topic's main characteristics

Label:"""
    else:
        prompt = """Analyze this fishing survey topic and create a descriptive, specific label (5-15 words).

Keywords: [KEYWORDS]
Examples: [DOCUMENTS]

The label must be:
- Specific and descriptive (e.g., "Boat Launch Facilities and Access Points" not "Fishing Access")
- Include relevant context from keywords and examples
- Clear and professional
- Focused on the unique aspect of this topic
- Informative enough to understand the topic's main characteristics

Label:"""
    
    representation_model = TextGeneration(
        generator,
        prompt=prompt,
        nr_docs=15,  # Increased from 10 to show more examples
        diversity=0.3,
        doc_length=150  # Increased from 100 to show more context per document
    )
    
    print(f"\n✅ Offline LLM ready for topic labeling")
    print(f"   Model: {selected_config['description']}")
    print(f"   Device: {device_name}")
    print(f"   Tokenizer: Default (custom tokenizer removed to fix bug)")
    print(f"   Fully reproducible: {seed}")
    
    return representation_model

def create_bertopic_model(seed: int = 42,
                         min_cluster_size: int = 3,
                         umap_n_neighbors: int = 15,
                         umap_n_components: int = 5,
                         nr_topics: Optional[int] = None,
                         top_n_words: int = 15,
                         n_gram_range: Tuple[int, int] = (1, 4)) -> BERTopic:
    """Create BERTopic model with descriptive labels."""
    from sklearn.feature_extraction.text import CountVectorizer
    from sentence_transformers import SentenceTransformer
    from umap import UMAP
    from hdbscan import HDBSCAN
    from bertopic.representation import KeyBERTInspired, MaximalMarginalRelevance
    
    embedding_model = SentenceTransformer('all-MiniLM-L6-v2')
    
    umap_model = UMAP(
        n_neighbors=umap_n_neighbors,
        n_components=umap_n_components,
        min_dist=0.0,
        metric='cosine',
        random_state=seed
    )
    
    hdbscan_model = HDBSCAN(
        min_cluster_size=min_cluster_size,
        metric='euclidean',
        cluster_selection_method='eom',
        prediction_data=True
    )
    
    vectorizer_model = CountVectorizer(
        stop_words="english",
        ngram_range=n_gram_range,
        min_df=2
    )
    
    keybert_model = KeyBERTInspired()
    mmr_model = MaximalMarginalRelevance(diversity=0.3)
    
    representation_model = {
        "KeyBERT": keybert_model,
        "MMR": mmr_model,
    }
    
    topic_model = BERTopic(
        language="english",
        calculate_probabilities=True,
        verbose=True,
        embedding_model=embedding_model,
        umap_model=umap_model,
        hdbscan_model=hdbscan_model,
        vectorizer_model=vectorizer_model,
        representation_model=representation_model,
        nr_topics=nr_topics,
        top_n_words=top_n_words
    )
    
    print(f"\n✅ BERTopic model created with:")
    print(f"   - KeyBERT + MMR representation")
    print(f"   - min_cluster_size: {min_cluster_size}")
    print(f"   - umap_n_neighbors: {umap_n_neighbors}")
    print(f"   - umap_n_components: {umap_n_components}")
    print(f"   - nr_topics: {nr_topics if nr_topics else 'Auto'}")
    print(f"   - top_n_words: {top_n_words}")
    print(f"   - n_gram_range: {n_gram_range}")
    
    return topic_model

def display_topic_summary(topic_model, topics: List[int], responses: List[str],
                          top_n: int = 10) -> None:
    """Display formatted summary of topics with examples."""
    topic_info = topic_model.get_topic_info()
    topic_info = topic_info[topic_info['Topic'] != -1].sort_values('Count', ascending=False)
    
    print("=" * 80)
    print("TOP TOPICS")
    print("=" * 80)
    
    for idx, (_, row) in enumerate(topic_info.head(top_n).iterrows(), 1):
        topic_id = row['Topic']
        count = row['Count']
        percentage = (count / len(responses)) * 100
        
        print(f"\n📌 Rank #{idx} | {row['Name']} ({count} responses, {percentage:.1f}%)")
        
        keywords = [word for word, _ in topic_model.get_topic(topic_id)[:8]]
        print(f"   Keywords: {', '.join(keywords)}")
        
        topic_docs = [responses[i] for i, t in enumerate(topics) if t == topic_id][:3]
        print(f"   Examples:")
        for doc in topic_docs:
            print(f"     • {doc[:90]}{'...' if len(doc) > 90 else ''}")
        print("-" * 80)

def print_model_statistics(topics: List[int], responses: List[str]) -> None:
    """Print statistics about the topic modeling results."""
    n_topics = len(set(topics)) - 1
    n_outliers = sum(1 for t in topics if t == -1)
    outlier_pct = (n_outliers / len(responses)) * 100
    
    print("\n" + "=" * 60)
    print("MODEL RESULTS SUMMARY")
    print("=" * 60)
    print(f"Total responses: {len(responses)}")
    print(f"Topics identified: {n_topics}")
    print(f"Outliers: {n_outliers} ({outlier_pct:.1f}%)")
    
    if outlier_pct > 20:
        print(f"\n⚠️ Warning: High outlier rate ({outlier_pct:.1f}%)")
        print("   Consider adjusting min_cluster_size or UMAP parameters")
    elif outlier_pct < 5:
        print(f"\n✅ Excellent: Low outlier rate ({outlier_pct:.1f}%)")
    else:
        print(f"\n✅ Good: Acceptable outlier rate ({outlier_pct:.1f}%)")
    
    print("=" * 60)

def save_results(topic_model, topics: List[int], responses: List[str],
                 model_name: str = "bertopic_model") -> None:
    """Save model and results to files."""
    model_path = f"{model_name}_{SEED}"
    topic_model.save(model_path, serialization="safetensors", save_ctfidf=True)
    print(f"✅ Model saved to: {model_path}")
    
    topic_info = topic_model.get_topic_info()
    topic_labels = {}
    for _, row in topic_info.iterrows():
        topic_labels[row['Topic']] = row['Name']
    
    results_df = pd.DataFrame({
        'response': responses,
        'topic_id': topics,
        'topic_label': [topic_labels.get(t, 'Outlier') for t in topics]
    })
    results_df.to_csv(f'{model_name}_assignments.csv', index=False)
    print(f"✅ Topic assignments saved to: {model_name}_assignments.csv")
    
    topic_info.to_csv(f'{model_name}_summary.csv', index=False)
    print(f"✅ Topic summary saved to: {model_name}_summary.csv")

print("✅ All helper functions loaded successfully!")

✅ All helper functions loaded successfully!


## Data Loading and Preprocessing

In [3]:
# Load the survey data
try:
    df = pd.read_csv('test_survey.csv')
    
    if df.shape[1] < 1:
        raise ValueError("CSV must have at least one column")
    
    print(f"✅ Dataset loaded successfully")
    print(f"   Shape: {df.shape}")
    print(f"\n   Column name: {df.columns[0]}")
    
except FileNotFoundError:
    print("❌ Error: test_survey.csv not found in current directory")
    raise
except Exception as e:
    print(f"❌ Error loading CSV: {e}")
    raise

print("\nFirst few rows:")
df.head()

✅ Dataset loaded successfully
   Shape: (203, 1)

   Column name: Q7_other: How much does each of the following factors attract you to go fishing at a particular site?

First few rows:


,Q7_other: How much does each of the following factors attract you to go fishing at a particular site?
0,10 mph lake speed limits
1,ability for family fishing
2,ability to catch and KEEP fish species present.
3,Access Boat ramps available
4,access to fish


In [4]:
# Extract and preprocess responses
raw_responses = df.iloc[:, 0].dropna().astype(str).tolist()
raw_responses = raw_responses[1:]  # Remove header

print(f"Raw responses extracted: {len(raw_responses)}\n")
analyze_response_statistics(raw_responses)

Raw responses extracted: 202


📊 Response Statistics:
   Total responses: 202

   Word count per response:
      Mean: 4.4
      Median: 3
      Min: 1
      Max: 52

   Character count per response:
      Mean: 25.0
      Median: 18

   ⚠️ Found 7 duplicate responses:
      'ease of access...' appears 3 times
      'quality of fish...' appears 3 times
      'weather...' appears 3 times
      'close to home...' appears 3 times
      'time off...' appears 2 times


In [5]:
# Preprocess responses
responses, preprocessing_stats = preprocess_responses(
    raw_responses,
    min_length=2,
    remove_duplicates=False,
    verbose=True
)

analyze_response_statistics(responses)

DATA PREPROCESSING SUMMARY
Original responses: 202
Empty/invalid removed: 0
Too short (< 2 words) removed: 22

✅ Final response count: 180
   Retention rate: 89.1%

📊 Response Statistics:
   Total responses: 180

   Word count per response:
      Mean: 4.9
      Median: 3
      Min: 2
      Max: 52

   Character count per response:
      Mean: 27.1
      Median: 20

   ⚠️ Found 6 duplicate responses:
      'ease of access...' appears 3 times
      'close to home...' appears 3 times
      'quality of fish...' appears 3 times
      'time off...' appears 2 times
      'camping available...' appears 2 times


In [6]:
# Display sample responses
print("\n📝 Sample of processed responses:")
for i, resp in enumerate(responses[:10], 1):
    print(f"{i:2d}. {resp}")


📝 Sample of processed responses:
 1. ability for family fishing
 2. ability to catch and KEEP fish species present.
 3. Access Boat ramps available
 4. access to fish
 5. amount of fish in water
 6. appropriate for children, safe
 7. availability for disabled fishermen.
 8. availability for RV sites
 9. being able to keep some fish to feed my family
10. being able to use barbed hooks to catch my fish and not barbless in order to loose my fish because when fighting a fish using a barbless hook and in such case loosing so called fish it will then be tired and has a greater chance of being eaten by a seal


# Create BERTopic model with optimal number of topics from Elbow Method
topic_model = create_bertopic_model(
    seed=SEED,
    min_cluster_size=3,
    umap_n_neighbors=15,
    umap_n_components=5,
    nr_topics=optimal_nr_topics  # Use optimal number from Elbow Method
)

# Fit the model
print("\n🔄 Training BERTopic model...\n")
topics, probs = topic_model.fit_transform(responses)

# Print statistics
print_model_statistics(topics, responses)

In [7]:
# Create BERTopic model with capped number of topics
topic_model = create_bertopic_model(
    seed=SEED,
    min_cluster_size=3,
    umap_n_neighbors=15,
    umap_n_components=5,
    nr_topics=8,  # Cap the number of topics to 8
    top_n_words=15,  # More words for descriptive labels
    n_gram_range=(1, 4)  # Include up to 4-grams for better phrases
)

# Fit the model
print("\n🔄 Training BERTopic model...\n")
topics, probs = topic_model.fit_transform(responses)

# Print statistics
print_model_statistics(topics, responses)

2025-11-19 20:12:31,422 - BERTopic - Embedding - Transforming documents to embeddings.



✅ BERTopic model created with:
   - KeyBERT + MMR representation
   - min_cluster_size: 3
   - umap_n_neighbors: 15
   - umap_n_components: 5
   - nr_topics: 8
   - top_n_words: 15
   - n_gram_range: (1, 4)

🔄 Training BERTopic model...



Batches:   0%|          | 0/6 [00:00<?, ?it/s]

2025-11-19 20:12:32,653 - BERTopic - Embedding - Completed ✓
2025-11-19 20:12:32,654 - BERTopic - Dimensionality - Fitting the dimensionality reduction algorithm
OMP: Info #276: omp_set_nested routine deprecated, please use omp_set_max_active_levels instead.
2025-11-19 20:12:37,916 - BERTopic - Dimensionality - Completed ✓
2025-11-19 20:12:37,918 - BERTopic - Cluster - Start clustering the reduced embeddings
2025-11-19 20:12:37,937 - BERTopic - Cluster - Completed ✓
2025-11-19 20:12:37,938 - BERTopic - Representation - Extracting topics using c-TF-IDF for topic reduction.
2025-11-19 20:12:37,954 - BERTopic - Representation - Completed ✓
2025-11-19 20:12:37,955 - BERTopic - Topic reduction - Reducing number of topics
2025-11-19 20:12:37,959 - BERTopic - Representation - Fine-tuning topics using representation models.
2025-11-19 20:12:39,791 - BERTopic - Representation - Completed ✓
2025-11-19 20:12:39,792 - BERTopic - Topic reduction - Reduced number of topics from 18 to 8



MODEL RESULTS SUMMARY
Total responses: 180
Topics identified: 7
Outliers: 34 (18.9%)

✅ Good: Acceptable outlier rate (18.9%)


In [8]:
# View initial topics
topic_info = topic_model.get_topic_info()

print("Initial Topics (before LLM labeling):")
print(topic_info[['Topic', 'Count', 'Name']].to_string())

print("\n" + "=" * 60)
print("Topic distribution:")
print(pd.Series(topics).value_counts().sort_index())

Initial Topics (before LLM labeling):
   Topic  Count                                          Name
0     -1     34                      -1_family_just_wild_fish
1      0     66                      0_fish_fishing_good_year
2      1     33  1_access_ease_disabled_availability disabled
3      2     19                    2_river_home_live_outdoors
4      3     10                       3_time_family_away_just
5      4      7                4_catch_release_large_friendly
6      5      7              5_weather_season_conditions_fall
7      6      4                             6_guide_service__

Topic distribution:
-1    34
 0    66
 1    33
 2    19
 3    10
 4     7
 5     7
 6     4
Name: count, dtype: int64


## Offline LLM Topic Labeling

This cell loads an offline LLM and uses it to generate high-quality topic labels.

**Model Selection:**
- `"auto"` - Automatically tries models in order (recommended)
- `"flan-t5"` - Google Flan-T5-Large (~3GB, reliable)
- `"flan-t5-base"` - Google Flan-T5-Base (~1GB, faster)
- `"llama"` - Meta Llama 3.2 1B (~5GB, best quality)
- `"phi"` - Microsoft Phi-3 Mini (~7GB, very good)

In [ ]:
# Setup offline LLM for topic labeling
print("\n" + "=" * 70)
print("SETTING UP OFFLINE LLM FOR TOPIC LABELING")
print("=" * 70)

# Choose model: "auto", "flan-t5", "flan-t5-base", "llama", or "phi"
MODEL_CHOICE = "llama"  # Using Llama for best quality labels

representation_model = setup_offline_llm(model_choice=MODEL_CHOICE, seed=SEED)

print("\n" + "=" * 70)


SETTING UP OFFLINE LLM FOR TOPIC LABELING
✅ Using Apple Silicon GPU (Metal) for acceleration

🔄 Loading Meta Llama 3.2 1B (best quality, ~5GB)...


In [ ]:
# Generate labels using offline LLM
print("\n🔄 Generating topic labels with offline LLM...")
print("This may take a few minutes depending on your hardware.\n")

# Update topics with LLM-generated labels
topic_model.update_topics(responses, representation_model=representation_model)

print("\n✅ Topic labels generated successfully!")


🔄 Generating topic labels with offline LLM...
This may take a few minutes depending on your hardware.



  0%|          | 0/18 [00:00<?, ?it/s]


UnboundLocalError: cannot access local variable 'truncated_document' where it is not associated with a value

## Results Analysis

In [ ]:
# Generate labels using offline LLM - DEBUGGED VERSION
print("\n🔄 Generating topic labels with offline LLM...")
print("This may take a few minutes depending on your hardware.\n")

try:
    # Update topics with LLM-generated labels
    topic_model.update_topics(
        responses,
        representation_model=representation_model
    )

    print("\n✅ Topic labels generated successfully!")

    # Display updated labels
    topic_info = topic_model.get_topic_info()
    print("\n📊 Updated Topic Labels (After LLM):")
    print("=" * 80)
    print(topic_info[['Topic', 'Count', 'Name']].to_string())
    print("=" * 80)

except Exception as e:
    print(f"\n❌ Error during LLM labeling: {type(e).__name__}: {e}")
    print("\n⚠️ Using keyword-based labels instead")
    print("   Debug info: Check BERTopic version and tokenizer compatibility")

    # Show current labels as fallback
    topic_info = topic_model.get_topic_info()
    print("\n📊 Current Topic Labels (c-TF-IDF):")
    print(topic_info[['Topic', 'Count', 'Name']].to_string())

In [ ]:
# Bar chart of top topics
fig_bar = topic_model.visualize_barchart(top_n_topics=10, height=400)
fig_bar.show()

print("\n✅ Bar chart displayed above")

# Intertopic distance map
fig_map = topic_model.visualize_topics()
fig_map.show()

print("\n✅ Distance map displayed above")

In [ ]:
# Bar chart of top topics
fig_bar = topic_model.visualize_barchart(top_n_topics=10, height=400)
fig_bar.show()

fig_bar.write_html("topic_barchart_offline_llm.html")
print("\n✅ Bar chart saved to: topic_barchart_offline_llm.html")

In [ ]:
# Intertopic distance map
fig_map = topic_model.visualize_topics()
fig_map.show()

fig_map.write_html("topic_distance_map_offline_llm.html")
print("\n✅ Distance map saved to: topic_distance_map_offline_llm.html")

# Visualize hierarchy
fig_hierarchy = topic_model.visualize_hierarchy(hierarchical_topics=hierarchical_topics)
fig_hierarchy.show()

print("\n✅ Hierarchy visualization displayed above")

In [ ]:
# Build hierarchical topics
print("🔄 Building hierarchical topic structure...\n")
hierarchical_topics = topic_model.hierarchical_topics(responses)

print(f"✅ Hierarchy created with {len(hierarchical_topics)} merges")

In [ ]:
# Visualize hierarchy
fig_hierarchy = topic_model.visualize_hierarchy(hierarchical_topics=hierarchical_topics)
fig_hierarchy.show()

fig_hierarchy.write_html("topic_hierarchy_offline_llm.html")
print("\n✅ Hierarchy visualization saved to: topic_hierarchy_offline_llm.html")

In [ ]:
# Show hierarchical merging structure
hierarchy_df = hierarchical_topics.copy().sort_values('Distance')

print("\nHierarchical Merging Structure:")
print("(Topics merge together as you move up the hierarchy)\n")
print(hierarchy_df[['Parent_ID', 'Topics', 'Distance']].head(20).to_string())

print("\nNote: Lower distances = more similar topics that merge first")

## Label Quality Comparison

In [ ]:
# Compare LLM-generated labels with keyword-based labels
print("\n" + "=" * 80)
print("LABEL QUALITY COMPARISON")
print("LLM-Generated vs. Keyword-Based Labels")
print("=" * 80)

topic_info = topic_model.get_topic_info()
top_topics = topic_info[topic_info['Topic'] != -1].sort_values('Count', ascending=False).head(10)

for idx, (_, row) in enumerate(top_topics.iterrows(), 1):
    topic_id = row['Topic']
    llm_label = row['Name']
    
    # Get keyword-based label (top 3 keywords)
    keywords = [w for w, _ in topic_model.get_topic(topic_id)[:3]]
    keyword_label = ' '.join([w.capitalize() for w in keywords])
    
    print(f"\n{idx}. Topic {topic_id} ({row['Count']} docs)")
    print(f"   LLM Label:     {llm_label}")
    print(f"   Keywords:      {keyword_label}")
    
    # Show example
    topic_docs = [responses[i] for i, t in enumerate(topics) if t == topic_id][:1]
    if topic_docs:
        print(f"   Example:       {topic_docs[0][:70]}...")

print("\n" + "=" * 80)

## Summary & Model Comparison

### New Features in This Notebook

#### 1. Elbow Method for Optimal Topic Count
- **Automatically** determines the optimal number of topics using WCSS (Within-Cluster Sum of Squares)
- Tests multiple topic counts and identifies the "elbow point"
- Visualizes WCSS curve and outlier percentage
- **No more guessing** how many topics to use!

#### 2. Unique Label Constraints
- LLM prompts now include **anti-duplication instructions**
- Prevents the model from creating similar or duplicate category labels
- Ensures each topic has a **distinct, meaningful name**
- Optional: Pass existing labels to avoid conflicts

#### 3. Streamlined Workflow
- Removed HTML file saving (visualizations display inline only)
- Cleaner output directory
- Focus on analysis, not file management

### Offline LLM Performance

This notebook uses **offline language models** for topic labeling:

| Model | Size | Quality | Speed | Best For |
|-------|------|---------|-------|----------|
| **Flan-T5-Large** | ~3GB | ⭐⭐⭐⭐ | Fast | Default choice, reliable |
| **Flan-T5-Base** | ~1GB | ⭐⭐⭐ | Very Fast | Low memory systems |
| **Llama 3.2 1B** | ~5GB | ⭐⭐⭐⭐⭐ | Medium | Best quality labels |
| **Phi-3 Mini** | ~7GB | ⭐⭐⭐⭐⭐ | Fast | Good GPU, best quality |

### Advantages of This Approach

✅ **Data-driven topic count** - Elbow Method finds optimal number automatically  
✅ **Unique labels** - Anti-duplication constraints ensure distinct categories  
✅ **Free** - No API costs whatsoever  
✅ **Private** - All data stays on your machine  
✅ **Reproducible** - Deterministic with seed  
✅ **Offline** - No internet required after download  
✅ **Better than patterns** - Understands context and nuance  
✅ **Almost as good as GPT** - For focused tasks like topic labeling  

### Workflow Summary

1. **Data Preprocessing** - Clean and filter survey responses
2. **Elbow Method** - Find optimal number of topics (5-25 range)
3. **Topic Modeling** - Create BERTopic model with optimal topic count
4. **LLM Labeling** - Generate unique, high-quality labels with offline LLM
5. **Analysis & Visualization** - Explore topics, hierarchies, and distributions
6. **Export Results** - Save model and topic assignments to CSV

### Performance Notes

- **GPU highly recommended** (Apple Silicon MPS or NVIDIA CUDA)
- **Elbow Method**: Takes 3-5 minutes for full analysis (worth it!)
- **First run**: Model downloads (~1-7GB depending on choice)
- **Subsequent runs**: Uses cached model
- **Label generation time**: ~1-3 minutes for 20 topics

### Tips for Best Results

1. **Use Elbow Method** - Let data determine optimal topic count
2. **Use GPU** - 10-20x faster than CPU
3. **Flan-T5-Large** - Best default for most users
4. **Llama/Phi** - If you have the memory/GPU for best quality
5. **Adjust topic range** - Modify min_topics/max_topics in Elbow Method if needed
6. **Monitor label uniqueness** - Check final labels for any similarities

In [ ]:
# Save model and results
save_results(
    topic_model=topic_model,
    topics=topics,
    responses=responses,
    model_name="fishing_survey_bertopic_offline_llm"
)

print("\n✅ All results saved! You can load the model later with:")
print(f"   topic_model = BERTopic.load('fishing_survey_bertopic_offline_llm_{SEED}')")

## Summary & Model Comparison

### Offline LLM Performance

This notebook uses **offline language models** for topic labeling:

| Model | Size | Quality | Speed | Best For |
|-------|------|---------|-------|----------|
| **Flan-T5-Large** | ~3GB | ⭐⭐⭐⭐ | Fast | Default choice, reliable |
| **Flan-T5-Base** | ~1GB | ⭐⭐⭐ | Very Fast | Low memory systems |
| **Llama 3.2 1B** | ~5GB | ⭐⭐⭐⭐⭐ | Medium | Best quality labels |
| **Phi-3 Mini** | ~7GB | ⭐⭐⭐⭐⭐ | Fast | Good GPU, best quality |

### Advantages of Offline LLM Approach

✅ **Free** - No API costs whatsoever
✅ **Private** - All data stays on your machine
✅ **Reproducible** - Deterministic with seed
✅ **Offline** - No internet required after download
✅ **Better than patterns** - Understands context and nuance
✅ **Almost as good as GPT** - For focused tasks like topic labeling

### When to Use What

- **Use this (Offline LLM)** - Best balance of quality and cost
- **Use GPT-4o-mini** - Need absolute best labels, have budget
- **Use Pattern Matching** - Very simple/repetitive topics, need instant results

### Performance Notes

- **GPU highly recommended** (Apple Silicon MPS or NVIDIA CUDA)
- **First run**: Model downloads (~1-7GB depending on choice)
- **Subsequent runs**: Uses cached model
- **Label generation time**: ~1-3 minutes for 20 topics

### Tips for Best Results

1. **Use GPU** - 10-20x faster than CPU
2. **Flan-T5-Large** - Best default for most users
3. **Llama/Phi** - If you have the memory/GPU for best quality
4. **Increase `nr_docs`** - Use more example documents for complex topics
5. **Adjust prompts** - Customize for your specific domain